# 📝 과제 02: 의약품 상담 도우미의 검색 품질 진단

## 만들 결과

약국 체인 본사의 **의약품 상담 도우미**는 식약처 「e약은요」 안내문 394편을 검색해 답하는 RAG입니다. 지난달 운영 팀이 답변 모델 비용을 줄이려고 설정 두 가지를 바꾼 뒤로, 매장 약사들이 "함께 먹으면 안 되는 약을 물으면 안내문에서 확인할 수 없다는 답이 늘었다"고 알려 왔습니다. 두 변경 가운데 어느 쪽이 원인인지 평가로 찾고 고칩니다.

- 바꾸기 전 설정인 기준선(baseline, 설정 이름 `기준선`)과 지금 설정(`현재 운영`)으로 골든셋(정답셋) 6문항의 답변을 만듭니다.
- 실행마다 지표 6개를 동시에 계산해 설정 × 지표 종합 표를 만듭니다. 교안 01의 생성 지표 Answer Correctness(답변 정확도)·Faithfulness(답변 충실도)·Answer Relevancy(답변 관련성)와 교안 02의 검색 지표 Context Recall(컨텍스트 재현율)·Context Precision(컨텍스트 정밀도)·Context Relevancy(컨텍스트 관련성)입니다. 지표 이름은 커리큘럼 표기대로 Context Relevancy로 쓰고, ragas 클래스 이름은 `ContextRelevance`(코드의 지표 객체는 `context_relevance`)입니다.
- 합격선으로 질문마다 합격 여부를 세고(절대 판정), 기준선과의 차이 표를 교안 02 6-4의 판정 순서표로 읽어 고장 위치를 찾습니다(상대 판정). 검색 문서 원문으로 원인을 확인합니다.
- 진단이 가리킨 설정 하나만 되돌린 `고친 운영` 설정을 다시 평가해 남긴 변경까지 판단하고, 합격 질문 수와 진단을 함께 근거로 결정 기록을 남깁니다.

교안 02 6절의 종합 평가와 고장 위치 진단을 의약품 안내문에 적용합니다. 교안은 설정을 하나씩 바꿔 넣은 고장을 진단했고, 과제는 운영 중에 함께 바뀐 설정 두 가지를 하나씩 가려냅니다.

의학적 조언이 아니라 수업용 검색 자료이며, 실제 복용은 의사·약사와 상의해야 합니다.

**운영 팀이 바꾼 설정**

| 항목 | 기준선(바꾸기 전) | 현재 운영 |
|---|---|---|
| 검색 문서 수(`k`) | 4 | 4(변경 없음) |
| 답변 모델에 넘기는 본문(`char_limit`) | 검색 문서 본문 전체 | 검색 문서마다 앞 300자 |
| 답변 프롬프트(`chain`) | 안내문에 있는 내용만 2~4문장으로 | 안내문에 있는 내용만 2문장 이내로 |

**설정 하나만 바꿔야 원인을 가릴 수 있습니다.** 교안 02처럼 설정을 하나만 바꾸고 기준선과 비교하면, 떨어진 지표가 그 변경을 가리킵니다. 여기서 "설정 하나"는 위 표의 한 행이고, 답변 프롬프트를 바꾸는 것도 설정 하나를 바꾸는 것입니다. 현재 운영은 두 행(`char_limit`, 프롬프트)이 함께 바뀌어 기준선과 비교만 해서는 어느 변경 때문인지 바로 알 수 없습니다. 그래서 진단과 원인 확인으로 한 변경을 가려내 되돌리고, 가려낸 변경 하나만 되돌리고 남은 변경 하나만 다른 `고친 운영`을 기준선과 다시 비교해 그 변경도 따로 판단합니다.

## 준비: 데이터

- **자료:** `data/과제02/`의 의약품 안내문 검색 DB(`chroma/`, 394편)와 골든셋 6문항(`questions.json`). 수업용 검색 DB는 Chroma·text-embedding-3-large로 다시 만들었습니다(앞 단원 과제의 KURE-v1·Neo4j 구성과 다름). 안내문은 앞 단원 과제와 같은 394편이라, 그 과제의 가정대로 주의·상호작용은 안내문 뒤쪽에 있어 끝까지 읽어야 합니다. 질문마다 정답 답변(`reference`)과 정답 문서(`relevant_doc_ids`), 정답 답변의 주장별 근거 문장(`claims`)이 있습니다. 정답 문서가 2개인 질문이 3개입니다. 출처와 골든셋 설계는 [자료 설명](data/과제02/README.md)에 있습니다.
- **순서:** 본문 자르기 함수 -> 실행 만들기 -> 지표 6개 계산 -> 종합 표 -> 합격선 -> 기준선 차이 표와 진단 -> 원인 확인 -> 고친 설정 다시 평가 -> 결정 기록
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **다시 실행할 때:** 평가 셀이 네트워크 오류로 멈추면 그 셀만 다시 실행합니다. 결과 목록을 셀 안에서 새로 만들므로 점수가 중복으로 쌓이지 않습니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 저장소 최상위 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")  # 답변 모델. .env에 없으면 기본 모델
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 검색 질문 임베딩: 배포한 문서 벡터와 같은 모델·차원이어야 합니다
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 입력 길이를 미리 검사해 나누는 자동 처리를 끕니다
)
chroma_dir = data_dir / "과제02" / "chroma"

print("답변 모델:", model_name)

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

os.environ["RAGAS_DO_NOT_TRACK"] = "true"  # ragas가 익명 사용 통계를 외부 서버로 보내지 않게 합니다

# 평가용 LLM: ragas 0.4 지표는 LangChain 모델 대신 OpenAI SDK의 비동기 클라이언트를 받습니다
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-6-luna")  # gpt-5.6-luna처럼 버전에 소수점이 든 이름은 ragas 0.4.3에서 요청 오류가 납니다
eval_client = AsyncOpenAI()

KOREAN_RULE = "모든 출력 문장(질문·주장·이유)은 한국어로 작성하세요."

evaluator_llm = llm_factory(
    eval_model_name,
    client=eval_client,
    max_tokens=8192,            # 응답 길이 상한. 추론 모델은 생각하는 데 쓴 토큰도 여기에 들어가므로 기본값(1024)보다 크게 둡니다
    system_prompt=KOREAN_RULE,  # 이 평가용 LLM을 쓰는 모든 지표에 한국어 규칙이 적용됩니다
)

# Answer Relevancy·Answer Correctness가 문장 사이 의미 유사도를 계산할 때 쓰는 임베딩 모델
evaluator_embeddings = embedding_factory(
    "openai",
    model="text-embedding-3-large",  # ragas 0.4.3 임베딩은 차원 인자를 받지 않아 기본 3072차원을 씁니다
    client=eval_client,
)

print("평가용 LLM:", eval_model_name)

In [ ]:
from langchain_chroma import Chroma


def evidence_id(doc):
    """검색 문서의 문서 ID(메타데이터 source_id)를 반환합니다."""
    return doc.metadata["source_id"]


vector_store = Chroma(
    collection_name="drug_guides",
    persist_directory=str(chroma_dir),
    embedding_function=embedding_model,
    create_collection_if_not_exists=False,  # 배포한 컬렉션이 없으면 새로 만들지 않고 오류를 냅니다
)

# 문서 ID로 저장된 본문(제목 + 줄바꿈 + 안내문)을 찾는 dict: 다시 임베딩하지 않고 읽기만 합니다
stored_documents = vector_store.get(include=["documents", "metadatas"])
documents_by_id = {}
for text, meta in zip(stored_documents["documents"], stored_documents["metadatas"]):
    documents_by_id[meta["source_id"]] = text

print("의약품 안내문:", len(documents_by_id))

In [ ]:
# 골든셋 6문항 읽기
questions = json.loads((data_dir / "과제02" / "questions.json").read_text(encoding="utf-8"))

# 질문마다 한 행씩 표로 보기
question_rows = []
for entry in questions:
    question_rows.append({
        "질문 ID": entry["question_id"],
        "질문": entry["question"],
        "정답 답변": entry["reference"],
        "정답 문서": ", ".join(entry["relevant_doc_ids"]),
    })
display(pd.DataFrame(question_rows))

답변 프롬프트 두 개, 동시에 보낼 요청 수의 상한 `MAX_CONCURRENCY`, 설정표 `SETTINGS`입니다. 설정 하나는 검색 문서 수(`k`), 검색 문서마다 답변 모델에 넘길 글자 수(`char_limit`, 0이면 자르지 않음), 답변 체인(`chain`)으로 정해집니다. 뒤의 작성 셀은 모두 이 표를 읽어 설정을 적용합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 기본 프롬프트(바꾸기 전): 안내문에 있는 내용만 근거로 2~4문장
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 약국 체인의 의약품 상담 도우미입니다. 아래 의약품 안내문에 있는 내용만 근거로 2~4문장으로 답하세요. "
               "안내문에 없는 내용은 추측하지 말고 안내문에서 확인할 수 없다고 말하세요."),
    ("human", "안내문:\n{context}\n\n질문: {question}"),
])
answer_chain = answer_prompt | llm

# 운영 팀이 바꾼 프롬프트: 같은 규칙에서 답변 길이만 2문장 이내로 줄였습니다
short_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 약국 체인의 의약품 상담 도우미입니다. 아래 의약품 안내문에 있는 내용만 근거로 2문장 이내로 짧게 답하세요. "
               "안내문에 없는 내용은 추측하지 말고 안내문에서 확인할 수 없다고 말하세요."),
    ("human", "안내문:\n{context}\n\n질문: {question}"),
])
short_chain = short_prompt | llm

# 동시에 보낼 요청 수의 상한입니다. 한꺼번에 너무 많이 보내면 API 요청 한도(분당 요청·토큰 수)를 넘어 429 오류가 날 수 있습니다.
# 32는 ragas evaluate()의 기본 동시 실행 수(RunConfig의 max_workers, 16)보다 넉넉하게 잡은 값입니다. 429 오류가 나면 줄이고, 키 한도가 넉넉하면 늘립니다.
MAX_CONCURRENCY = 32

# 설정표: 검색 문서 수, 문서마다 넘길 글자 수(0이면 자르지 않음), 답변 체인
SETTINGS = {
    "기준선": {"k": 4, "char_limit": 0, "chain": answer_chain},
    "현재 운영": {"k": 4, "char_limit": 300, "chain": short_chain},
}


def format_contexts(doc_ids, contexts):
    """검색 문서 ID와 넘길 본문을 "[문서 ID] 본문" 블록으로 이어 붙인 문자열을 반환합니다."""
    blocks = []
    for doc_id, context in zip(doc_ids, contexts):
        blocks.append(f"[{doc_id}] {context}")
    return "\n\n".join(blocks)

## 1. 설정대로 검색 문서 본문을 자르는 함수를 만드세요

현재 운영 설정을 평가하려면 운영 중인 도우미가 답변 모델에 넘기는 본문을 그대로 재현해야 하므로, 설정표의 `char_limit`를 검색 문서에 적용하는 함수부터 만듭니다.

**작성:** `cut_contexts(docs, char_limit)` 함수

**목적:** 검색기가 돌려준 문서 목록을 평가 샘플의 검색 문서 칸(`retrieved_contexts`)에 넣을 문자열 목록으로 바꿉니다. 답변 모델에 넘기는 본문과 평가용 LLM이 읽는 검색 문서가 같은 문자열이어야 점수가 운영 상황을 나타냅니다.

**구현할 처리**
1. 검색 문서마다 본문(`page_content`)을 꺼냅니다.
2. `char_limit`가 0이면 본문 전체를, 0보다 크면 앞 `char_limit` 글자만 담습니다.
3. 검색 문서 순서를 바꾸지 않고 목록으로 반환합니다.

**반환값:** 본문 문자열 목록(검색 문서 수와 같은 길이)

In [ ]:
# [작성] 설정의 char_limit만큼 검색 문서 본문을 자른 목록을 반환하는 함수를 완성하세요.
def cut_contexts(docs, char_limit):
    """검색 문서마다 본문을 앞 char_limit 글자까지 자른 목록을 반환합니다. char_limit가 0이면 자르지 않습니다."""
    contexts = []
    for doc in docs:
        # 1~2. char_limit가 0이면 본문 전체, 아니면 앞 char_limit 글자를 contexts에 더하세요.
        ...

    # 3. 검색 문서 순서 그대로 반환
    ...

In [ ]:
# [자가채점] 길이가 다른 가짜 문서 두 개로 자르기와 순서를 확인합니다. API는 부르지 않습니다.
from langchain_core.documents import Document

long_text = "가" * 450 + "나" * 50
fake_docs = [Document(page_content=long_text, metadata={"source_id": "fake_long"}),
             Document(page_content="다" * 120, metadata={"source_id": "fake_short"})]

full = cut_contexts(fake_docs, 0)
assert isinstance(full, list), "본문 문자열 목록을 return 하세요. `...`를 채웠는지도 확인하세요."
assert full == [long_text, "다" * 120], "char_limit가 0이면 본문 전체를 문서 순서대로 담습니다."
assert cut_contexts(fake_docs, 300) == ["가" * 300, "다" * 120], "char_limit가 0보다 크면 앞 char_limit 글자만 담고, 그보다 짧은 본문은 그대로 둡니다."
assert cut_contexts(fake_docs, 460) == ["가" * 450 + "나" * 10, "다" * 120], "본문 뒤쪽이 아니라 앞쪽 char_limit 글자를 남기세요."
assert cut_contexts([], 300) == [], "검색 문서가 없으면 빈 목록을 반환합니다."
print("cut_contexts 확인 완료")

## 2. 설정마다 골든셋 6문항의 실행을 만드세요

설정 하나와 질문 하나의 조합을 교안 02처럼 **실행**이라고 부릅니다. 같은 질문을 설정마다 같은 방식으로 실행해야 지표 차이를 설정 차이로 읽을 수 있으므로, 설정 이름만 바꿔 다시 부를 수 있는 함수로 만듭니다. 8번에서 고친 설정도 이 함수로 실행합니다.

| 이 절에서 쓰는 코드 | 입력과 반환값 |
|---|---|
| `SETTINGS[설정 이름]` | `k`, `char_limit`, `chain` 세 키의 dict |
| `evidence_id(doc)` | 검색 문서의 문서 ID |
| `format_contexts(doc_ids, contexts)` | 답변 프롬프트의 `{context}`에 넣을 문자열 |

**작성:** `generate_runs(config_names)` 함수와 `runs`(기준선·현재 운영 두 설정의 실행 12개)

**목적:** 교안 02의 실행 만들기를 설정표로 일반화합니다. 답변은 질문마다 따로 부르지 않고 설정의 체인 `batch`로 6문항을 한 번에 받습니다.

**구현할 처리**
1. 설정의 `k`로 `vector_store.as_retriever(search_kwargs={"k": ...})` 검색기를 만듭니다.
2. 질문마다 검색하고, `cut_contexts`에 설정의 `char_limit`를 넘겨 답변 모델에 넘길 본문을 만듭니다.
3. 답변 입력 `{"question": 질문, "context": format_contexts(문서 ID 목록, 본문 목록)}`을 `inputs_list`에 쌓습니다. 골격의 `setting["chain"].batch(...)`가 설정의 답변 체인으로 6개를 한 번에 보내고, 결과는 입력 순서대로 돌아옵니다. `max_concurrency`에는 설정표 셀의 `MAX_CONCURRENCY`(동시에 보낼 요청 수의 상한)를 넘기고, 3번 평가도 같은 상한을 씁니다.
4. 실행 dict를 골격의 키를 이 순서로 써서 쌓습니다. `user_input`은 질문, `retrieved_contexts`는 2번에서 자른 본문 목록, `response`는 답변의 `.text`에서 앞뒤 공백을 지운 문자열, `reference`는 골든셋의 정답 답변입니다.

**반환값:** 실행 dict 목록. `config_names`의 설정 순서대로, 설정 안에서는 `questions` 순서대로 쌓습니다.

In [ ]:
# [작성] 설정마다 골든셋을 검색하고 답변을 batch로 만들어 실행 목록을 반환하는 함수를 완성하세요.
def generate_runs(config_names):
    """설정 이름 목록을 받아 설정마다 질문 6개의 실행(평가 샘플 네 칸과 설정·검색 문서 ID)을 담은 목록을 반환합니다."""
    runs = []
    for config in config_names:
        setting = SETTINGS[config]

        # 1. 설정의 k로 검색기를 만드세요.
        retriever = ...

        # 2. 질문마다 검색하고, 설정의 char_limit로 자른 본문과 답변 입력을 만드세요.
        ids_list = []
        contexts_list = []
        inputs_list = []
        for entry in questions:
            docs = retriever.invoke(entry["question"])
            doc_ids = [evidence_id(doc) for doc in docs]
            contexts = ...
            ids_list.append(doc_ids)
            contexts_list.append(contexts)

            # 3. 답변 입력(question, context)을 inputs_list에 더하세요.
            ...

        # 설정의 답변 체인으로 답변 6개를 batch로 한 번에 받습니다 (결과는 입력 순서대로 돌아옵니다)
        answers = setting["chain"].batch(
            inputs_list,
            config={"max_concurrency": MAX_CONCURRENCY},  # max_concurrency: 동시에 보낼 요청 수 상한(기본은 제한 없음에 가까운 스레드 풀 크기). 평가와 같은 상한을 씁니다
        )

        # 4. ragas 지표에 넘길 칸 이름으로 실행을 하나씩 쌓으세요.
        for entry, doc_ids, contexts, answer in zip(questions, ids_list, contexts_list, answers):
            runs.append({
                "config": config,
                "question_id": entry["question_id"],
                "user_input": ...,
                "retrieved_contexts": ...,
                "retrieved_ids": doc_ids,
                "response": ...,
                "reference": ...,
            })
    return runs


runs = generate_runs(["기준선", "현재 운영"])
print("실행 수:", len(runs))

In [ ]:
# [자가채점] 실행의 수·순서·키를 확인하고, 검색 문서 칸을 Chroma에 저장된 본문과 설정의 char_limit로 대조합니다. API는 다시 부르지 않습니다.
def check_runs(run_list, config_names):
    """실행 목록이 설정 순서·질문 순서·키·자른 본문 규칙을 지키는지 확인합니다."""
    run_keys = ["config", "question_id", "user_input", "retrieved_contexts", "retrieved_ids", "response", "reference"]
    assert isinstance(run_list, list), "실행 목록(list)을 return 하세요. `...`를 채웠는지도 확인하세요."
    assert len(run_list) == len(config_names) * len(questions), f"실행은 설정 {len(config_names)}개 × 질문 {len(questions)}개, 모두 {len(config_names) * len(questions)}개입니다."
    position = 0
    for config in config_names:
        char_limit = SETTINGS[config]["char_limit"]
        for entry in questions:
            run = run_list[position]
            position += 1
            assert list(run) == run_keys, f"실행 dict의 키는 {run_keys}를 이 순서로 씁니다."
            assert run["config"] == config and run["question_id"] == entry["question_id"], "실행은 config_names의 설정 순서대로, 설정 안에서는 questions 순서대로 쌓으세요."
            assert run["user_input"] == entry["question"], "user_input에는 골든셋의 질문을 넣으세요."
            assert run["reference"] == entry["reference"], "reference에는 골든셋의 정답 답변(reference)을 넣으세요."
            assert len(run["retrieved_ids"]) == SETTINGS[config]["k"], f"{config}: 검색 문서 수가 설정의 k와 다릅니다. 검색기를 설정의 k로 만드세요."
            assert len(run["retrieved_contexts"]) == len(run["retrieved_ids"]), "retrieved_contexts는 검색 문서마다 본문 하나입니다."
            for doc_id, context in zip(run["retrieved_ids"], run["retrieved_contexts"]):
                stored = documents_by_id[doc_id]
                if char_limit == 0:
                    assert context == stored, f"{config} {entry['question_id']}: char_limit가 0인 설정은 본문 전체를 넘깁니다."
                else:
                    assert context == stored[:char_limit], f"{config} {entry['question_id']}: 본문 앞 {char_limit}자를 cut_contexts로 잘라 넘기세요."
            assert isinstance(run["response"], str) and run["response"] != "", "response에는 답변의 .text를 앞뒤 공백을 지워 넣으세요."
            assert run["response"] == run["response"].strip(), "response는 .strip()으로 앞뒤 공백을 지우세요."
            assert run["response"] != run["reference"], "response에는 답변 모델이 만든 답변을 넣으세요. 정답 답변(reference)이 아닙니다."


check_runs(runs, ["기준선", "현재 운영"])
print("실행 확인 완료:", len(runs), "개")

In [ ]:
# 실행 12개를 한 표로 보기 (넘긴 본문 글자 수는 검색 문서 4개를 합친 길이, 답변은 앞 60자만)
preview_rows = []
for run in runs:
    preview_rows.append({
        "설정": run["config"],
        "질문 ID": run["question_id"],
        "검색 문서 ID": ", ".join(run["retrieved_ids"]),
        "넘긴 본문 글자 수": len("".join(run["retrieved_contexts"])),
        "답변 앞 60자": run["response"][:60],
    })
display(pd.DataFrame(preview_rows))

## 3. 실행마다 지표 6개를 동시에 계산하세요

검색 지표 셋은 고장이 검색 문서 쪽에 있는지, Faithfulness는 답변 생성 쪽에 있는지, Answer Correctness는 최종 답이 정답 답변과 맞는지, Answer Relevancy는 답변이 질문에서 엇나갔는지 알려 줍니다. 교안 02 6절처럼 지표 6개를 같은 실행에 모두 계산합니다. 8번에서 고친 설정도 같은 함수로 평가합니다.

| 이 절에서 쓰는 코드 | 입력과 반환값 |
|---|---|
| `answer_correctness`, `context_recall`, `context_precision`, `context_relevance`, `faithfulness`, `answer_relevancy` | 교안 01·02와 같은 지표 객체. `await 지표.ascore(...)`의 결과 `.value`가 점수 |
| `METRIC_NAMES` | 점수 행의 지표 이름 여섯 개. 교안 02 종합 표의 열 순서(최종 결과, 검색, 생성)와 같습니다 |
| `run_limited(task)` | 교안 02와 같은 함수. 세마포어 자리가 날 때까지 기다렸다가 작업 하나(`지표.ascore(...)`)를 실행하고 결과를 돌려줍니다. 동시에 실행되는 작업은 `MAX_CONCURRENCY`개까지입니다 |

**작성:** `score_run(run)`, `evaluate_runs(run_list)` 두 비동기 함수. 바로 아래 자가채점이 평가를 돌리기 전에 두 함수의 구조를 먼저 확인하고, 그다음 셀이 `score_rows = await evaluate_runs(runs)`로 실행 12개의 점수 행 72개를 만듭니다.

**목적:** 교안 02 따라하기의 `asyncio.gather`(실행 하나의 지표 여럿을 동시에)와 본실습의 `tqdm_asyncio.gather`(실행 전체를 한꺼번에)를 합칩니다. 실행마다 끝나기를 기다리지 않고 한꺼번에 계산합니다.

**구현할 처리**
1. `score_run`: 지표 6개의 `ascore`를 하나씩 `run_limited(...)`로 감싸 `METRIC_NAMES` 순서대로 `asyncio.gather`에 넘깁니다. 지표마다 넘기는 칸은 교안 01·02의 평가 셀과 같습니다.
    - Answer Correctness: 질문·답변·정답 답변
    - Context Recall: 질문·검색 문서·정답 답변
    - Context Precision: 질문·정답 답변·검색 문서
    - Context Relevancy: 질문·검색 문서
    - Faithfulness: 질문·답변·검색 문서
    - Answer Relevancy: 질문·답변
2. `score_run`: `METRIC_NAMES` 순서로 지표마다 `설정`, `질문 ID`, `지표`, `점수` 네 키를 이 순서로 쓴 행을 만들어 여섯 행의 목록을 반환합니다. 점수는 결과의 `.value`입니다.
3. `evaluate_runs`: 실행마다 `score_run(run)`을 만들어 `tqdm_asyncio.gather`로 한꺼번에 실행합니다(`desc="평가"`). `score_run`은 `run_limited`로 감싸지 않습니다. 바깥 `score_run`이 자리를 차지한 채 안쪽 `ascore`의 자리를 기다리면 서로 기다리다 멈출 수 있고, 안쪽 `ascore`를 하나씩 감싸야 전체 동시 실행 수가 `MAX_CONCURRENCY`를 넘지 않습니다.
4. `evaluate_runs`: 실행마다 받은 여섯 행을 실행 순서대로 한 목록에 이어 붙여 반환합니다.

**반환값:** `score_rows`(점수 행 dict 목록, 실행 수 × 6)

답변이 "안내문에서 확인할 수 없다"뿐이면 Faithfulness가 나눌 주장이 없어 NaN이 될 수 있습니다. 표의 평균은 NaN을 빼고 계산합니다.

In [ ]:
import asyncio

from ragas.metrics.collections import (AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall,
                                       ContextRelevance, Faithfulness)

# 교안 01의 생성 지표 셋과 교안 02의 검색 지표 셋 (평가용 LLM·임베딩 모델을 함께 씁니다)
answer_correctness = AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings)  # 답변의 사실이 정답 답변과 맞는 정도
context_recall = ContextRecall(llm=evaluator_llm)        # 정답 답변의 주장 가운데 검색 문서로 뒷받침되는 비율
context_precision = ContextPrecision(llm=evaluator_llm)  # 검색 문서마다 정답 답변을 만드는 데 쓸모 있는지 판정하고 순위를 반영
context_relevance = ContextRelevance(llm=evaluator_llm)  # 검색 문서 묶음과 질문의 관련성을 0·1·2점으로 두 번 매겨 평균
faithfulness = Faithfulness(llm=evaluator_llm)          # 답변의 주장 가운데 검색 문서로 뒷받침되는 비율
answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)  # 답변에서 거꾸로 만든 질문과 원래 질문의 유사도

# 점수 행의 지표 이름 (종합 표의 열 순서: 최종 결과, 검색, 생성)
METRIC_NAMES = ["Answer Correctness", "Context Recall", "Context Precision", "Context Relevancy", "Faithfulness", "Answer Relevancy"]

# gather는 넘긴 작업을 한꺼번에 시작하므로, 세마포어로 동시에 실행되는 평가 작업 수를 MAX_CONCURRENCY개로 묶습니다
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)  # 동시에 실행 중인 작업이 MAX_CONCURRENCY개가 되면 나머지는 자리가 날 때까지 기다립니다


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행하고 결과를 돌려줍니다."""
    async with semaphore:
        return await task

In [ ]:
# [작성] 실행 하나의 지표 6개를 동시에 계산하는 함수와 실행 목록 전체를 한꺼번에 평가하는 함수를 완성하세요.
from tqdm.asyncio import tqdm_asyncio


async def score_run(run):
    """실행 하나의 지표 6개를 asyncio.gather로 동시에 계산해 점수 행 6개의 목록을 반환합니다."""
    # 1. 지표 6개의 ascore를 METRIC_NAMES 순서로 하나씩 run_limited로 감싸 동시에 실행하세요 (결과는 넘긴 순서대로 돌아옵니다)
    results = await asyncio.gather(
        run_limited(...),  # answer_correctness.ascore: 질문·답변·정답 답변
        run_limited(...),  # context_recall.ascore: 질문·검색 문서·정답 답변
        run_limited(...),  # context_precision.ascore: 질문·정답 답변·검색 문서
        run_limited(...),  # context_relevance.ascore: 질문·검색 문서
        run_limited(...),  # faithfulness.ascore: 질문·답변·검색 문서
        run_limited(...),  # answer_relevancy.ascore: 질문·답변
    )

    # 2. METRIC_NAMES 순서로 지표마다 한 행씩 만드세요.
    rows = []
    for metric_name, result in zip(METRIC_NAMES, results):
        ...
    return rows


async def evaluate_runs(run_list):
    """실행 목록 전체를 한꺼번에 평가해 점수 행을 실행 순서대로 이어 붙인 목록을 반환합니다."""
    # 3. 실행마다 score_run을 만들어 tqdm_asyncio.gather로 한꺼번에 실행하세요 (score_run은 run_limited로 감싸지 않습니다)
    results = ...

    # 4. 실행마다 받은 행 6개를 한 목록으로 이어 붙이세요.
    score_rows = []
    for rows in results:
        ...
    return score_rows

In [ ]:
# [자가채점] 비싼 평가를 돌리기 전에 두 함수의 구조만 확인합니다. API는 부르지 않습니다.
import inspect

assert inspect.iscoroutinefunction(score_run) and inspect.iscoroutinefunction(evaluate_runs), "score_run과 evaluate_runs는 async def로 만드세요."
score_source = inspect.getsource(score_run)
evaluate_source = inspect.getsource(evaluate_runs)
assert "run_limited(" in score_source, "score_run 안에서 지표의 ascore를 run_limited(...)로 감싸세요."
assert "run_limited(" not in evaluate_source, "evaluate_runs에서는 score_run을 run_limited로 감싸지 않습니다(바깥까지 감싸면 서로 기다리다 멈출 수 있습니다)."

# 지표 객체 6개의 ascore가 METRIC_NAMES 순서로 한 번씩 나오는지 (같은 지표 반복, 순서 바꿈을 막습니다)
metric_objects = ["answer_correctness", "context_recall", "context_precision", "context_relevance", "faithfulness", "answer_relevancy"]
positions = []
for name in metric_objects:
    call = name + ".ascore("
    assert score_source.count(call) == 1, f"score_run 안에 {call}...)를 한 번씩 쓰세요. 지표 6개를 모두, 같은 지표는 한 번만 넘깁니다."
    positions.append(score_source.find(call))
assert positions == sorted(positions), "지표 6개의 ascore는 METRIC_NAMES 순서(Answer Correctness, Context Recall, Context Precision, Context Relevancy, Faithfulness, Answer Relevancy)로 넘기세요."
print("평가 함수 구조 확인 완료")

In [ ]:
# 실행 12개를 지표 6개로 평가합니다 (평가용 LLM을 호출합니다)
score_rows = await evaluate_runs(runs)
print("점수 행:", len(score_rows))

In [ ]:
# [자가채점] 점수 행의 수·순서·키와 점수 범위(0~1 또는 NaN)를 확인합니다. LLM 점수 값 자체는 실행마다 달라 채점하지 않습니다.
import math


def check_score_rows(row_list, run_list):
    """점수 행이 실행 순서·지표 순서·키·점수 범위를 지키는지 확인하고 NaN 칸 수를 반환합니다."""
    assert isinstance(row_list, list), "점수 행 목록(list)을 return 하세요. `...`를 채웠는지도 확인하세요."
    metric_count = len(METRIC_NAMES)
    assert len(row_list) == len(run_list) * metric_count, f"점수 행은 실행 {len(run_list)}개 × 지표 {metric_count}개, 모두 {len(run_list) * metric_count}개입니다. 4번에서 행 {metric_count}개를 이어 붙였는지 확인하세요."
    nan_count = 0
    for position, run in enumerate(run_list):
        for offset, metric_name in enumerate(METRIC_NAMES):
            row = row_list[position * metric_count + offset]
            assert isinstance(row, dict) and list(row) == ["설정", "질문 ID", "지표", "점수"], "점수 행의 키는 설정, 질문 ID, 지표, 점수를 이 순서로 씁니다."
            assert row["설정"] == run["config"] and row["질문 ID"] == run["question_id"], "점수 행은 실행 순서대로 이어 붙이세요(tqdm_asyncio.gather는 넘긴 순서대로 결과를 돌려줍니다)."
            assert row["지표"] == metric_name, "실행 하나의 여섯 행은 METRIC_NAMES 순서이고, gather에 넘긴 지표 순서도 같아야 합니다."
            assert isinstance(row["점수"], float), "점수에는 ascore 결과의 .value(실수)를 넣으세요."
            if math.isnan(row["점수"]):
                nan_count += 1
            else:
                assert 0 <= row["점수"] <= 1, "RAGAS 점수는 0~1 사이입니다."
    assert nan_count < len(row_list), "모든 칸이 NaN입니다. 키와 평가용 LLM 설정부터 확인하세요."
    return nan_count


nan_count = check_score_rows(score_rows, runs)
print("점수 행 확인 완료:", len(score_rows), "행 | NaN 칸:", nan_count)

## 4. 설정 × 지표 종합 표를 만드세요

합격선 판정과 기준선과의 비교는 모두 설정마다 질문 6개의 평균을 읽으므로, 점수 행을 설정 × 지표 평균 표로 모읍니다.

**작성:** `overall_table`

**목적:** 교안 02의 종합 표처럼 행은 설정, 열은 지표인 평균 표를 만듭니다. 열은 최종 결과(Answer Correctness), 검색, 생성 순서로 읽으므로 `METRIC_NAMES` 순서로 둡니다. 6번의 진단과 결정 기록이 이 표의 숫자를 씁니다.

**구현할 처리**
1. `score_rows`로 DataFrame을 만듭니다.
2. `pivot_table`로 행(`index`)은 `설정`, 열(`columns`)은 `지표`, 값은 `점수`의 평균(`aggfunc="mean"`)인 표를 만듭니다. NaN 칸은 평균에서 빠집니다.
3. 열을 `METRIC_NAMES` 순서로 고르고(`표[METRIC_NAMES]`) 소수 둘째 자리로 반올림합니다.

**반환값:** `overall_table`(행 2개 × 열 6개 DataFrame)

In [ ]:
# [작성] 행은 설정, 열은 지표(METRIC_NAMES 순서), 칸은 질문 6개의 평균인 종합 표를 만드세요.
overall_table = ...
display(overall_table)

In [ ]:
# [자가채점] 표 모양과 칸 값을 score_rows에서 다시 계산한 평균과 대조합니다.
def check_table(table, row_list, config_names):
    """종합 표가 설정 × 지표 모양이고 칸마다 점수 평균(소수 둘째 자리)과 같은지 확인합니다."""
    assert isinstance(table, pd.DataFrame), "pivot_table 결과(DataFrame)를 담으세요. `...`를 채웠는지도 확인하세요."
    assert sorted(table.index) == sorted(config_names), f"행(index)은 설정 {config_names}입니다."
    assert list(table.columns) == METRIC_NAMES, "열(columns)은 지표 여섯 개를 METRIC_NAMES 순서로 둡니다(표[METRIC_NAMES])."
    expected = pd.DataFrame(row_list).groupby(["설정", "지표"])["점수"].mean().round(2)
    for (config, metric_name), value in expected.items():
        cell = table.loc[config, metric_name]
        if pd.isna(value):
            assert pd.isna(cell), f"{config} {metric_name}: 모든 질문이 NaN이면 칸도 NaN입니다."
        else:
            assert abs(cell - value) < 1e-9, f"{config} {metric_name}: 칸은 질문 6개 점수의 평균을 소수 둘째 자리로 반올림한 값입니다(aggfunc='mean', round(2))."


check_table(overall_table, score_rows, ["기준선", "현재 운영"])
print("종합 표 확인 완료")

아래 표는 같은 점수를 질문별로 펼친 것입니다. 평균이 크게 움직인 지표가 있으면 어느 질문에서 움직였는지 이 표에서 찾습니다.

In [ ]:
# 설정·질문별로 지표 6개의 점수를 펼친 표 (열은 METRIC_NAMES 순서)
display(pd.DataFrame(score_rows).pivot_table(index=["설정", "질문 ID"], columns="지표", values="점수")[METRIC_NAMES].round(2))

## 5. 합격선으로 합격한 질문 수를 세세요

평균만 보면 한 질문의 큰 실패가 가려지고, 기준선과 비교만 하면 기준선 자체가 부족한지 알 수 없습니다. 그래서 교안 02 6-3처럼 지표마다 정한 합격선(절대 판정)을 넘은 질문 수를 설정마다 셉니다.

| 이 절에서 쓰는 코드 | 입력과 반환값 |
|---|---|
| `THRESHOLDS` | 지표 이름과 합격선의 dict. 교안 02 6-3의 예시값이고, Answer Relevancy는 절댓값 합격선이 없어 넣지 않았습니다 |

**작성:** `count_passes(row_list)` 함수와 `pass_table`

**목적:** 8번에서 세 설정을 다시 셀 때도 쓰도록 함수로 만듭니다.

**구현할 처리**
1. `row_list`의 점수 행 가운데 지표가 `THRESHOLDS`에 있는 행만 씁니다.
2. 점수가 합격선 이상이면 1, 아니면 0인 `합격` 값을 `int(점수 >= 합격선)`으로 만들고, `설정`, `지표`, `합격` 세 키를 이 순서로 쓴 행을 쌓습니다. NaN은 비교 결과가 `False`라 0으로 셉니다.
3. `pivot_table`로 행은 `설정`, 열은 `지표`, 칸은 `합격`의 합(`aggfunc="sum"`)인 표를 만들고, 열을 `THRESHOLDS`의 순서(`list(THRESHOLDS)`)로 골라 반환합니다.

**반환값:** 설정 × 지표 5개 표(칸은 질문 6개 가운데 합격선을 넘은 질문 수)

In [ ]:
# 교안 02 6-3의 합격선. Answer Relevancy는 절댓값 합격선을 두지 않습니다
THRESHOLDS = {
    "Answer Correctness": 0.6,   # 정답 답변 주장의 절반 이상을 맞힌 수준
    "Context Recall": 0.8,       # 정답 답변 주장 2~3개 가운데 하나라도 빠지면 미달
    "Context Precision": 0.7,    # 1위 문서가 쓸모 있어야 넘는 값
    "Context Relevancy": 0.75,   # 평가용 LLM이 두 번 매긴 점수 가운데 한 번은 2점, 다른 한 번은 1점 이상이어야 넘는 값
    "Faithfulness": 0.8,         # 답변 주장 4개 가운데 하나만 근거가 없어도 미달
}

In [ ]:
# [작성] 설정마다 합격선을 넘은 질문 수를 세는 함수를 완성하세요.
def count_passes(row_list):
    """점수 행 목록으로 설정 × 지표마다 합격선을 넘은 질문 수 표를 반환합니다."""
    pass_rows = []
    for row in row_list:
        # 1~2. 합격선이 있는 지표만, 합격하면 1 아니면 0인 행을 쌓으세요.
        ...

    # 3. 설정 × 지표 합계 표를 THRESHOLDS 순서의 열로 반환하세요.
    ...


pass_table = count_passes(score_rows)
display(pass_table)

In [ ]:
# [자가채점] 가짜 점수 행으로 경계(합격선과 같은 점수)와 NaN을 확인하고, 실제 표는 score_rows에서 다시 센 값과 대조합니다.
fake_score_rows = [
    {"설정": "가", "질문 ID": "q1", "지표": "Context Recall", "점수": 0.8},             # 합격선과 같으면 합격
    {"설정": "가", "질문 ID": "q2", "지표": "Context Recall", "점수": 0.79},            # 합격선 아래
    {"설정": "가", "질문 ID": "q1", "지표": "Faithfulness", "점수": float("nan")},     # NaN은 0
    {"설정": "가", "질문 ID": "q2", "지표": "Faithfulness", "점수": 1.0},
    {"설정": "가", "질문 ID": "q1", "지표": "Answer Relevancy", "점수": 0.1},          # 합격선이 없는 지표는 세지 않음
    {"설정": "가", "질문 ID": "q1", "지표": "Answer Correctness", "점수": 0.6},
    {"설정": "가", "질문 ID": "q1", "지표": "Context Precision", "점수": 0.0},
    {"설정": "가", "질문 ID": "q1", "지표": "Context Relevancy", "점수": 0.75},
]
fake_table = count_passes(fake_score_rows)
assert isinstance(fake_table, pd.DataFrame), "pivot_table 결과(DataFrame)를 return 하세요. `...`를 채웠는지도 확인하세요."
assert list(fake_table.columns) == list(THRESHOLDS), "열은 THRESHOLDS의 지표 다섯 개를 그 순서로 둡니다. Answer Relevancy는 넣지 않습니다."
assert fake_table.loc["가", "Context Recall"] == 1, "합격은 점수가 합격선 이상(>=)일 때입니다. 0.8은 합격, 0.79는 불합격입니다."
assert fake_table.loc["가", "Faithfulness"] == 1, "NaN은 비교 결과가 False라 0으로 셉니다."
assert fake_table.loc["가", "Answer Correctness"] == 1 and fake_table.loc["가", "Context Relevancy"] == 1, "합격선과 같은 점수는 합격입니다."
assert fake_table.loc["가", "Context Precision"] == 0, "합격선보다 낮으면 0입니다."

assert sorted(pass_table.index) == ["기준선", "현재 운영"], "행(index)은 설정 두 개입니다."
for config in ["기준선", "현재 운영"]:
    for metric_name, threshold in THRESHOLDS.items():
        expected = 0
        for row in score_rows:
            if row["설정"] == config and row["지표"] == metric_name and row["점수"] >= threshold:
                expected += 1
        assert pass_table.loc[config, metric_name] == expected, f"{config} {metric_name}: 합격선 {threshold} 이상인 질문 수는 {expected}개입니다."
print("합격 표 확인 완료")

## 6. 기준선과의 차이 표를 만들고 판정 순서표로 진단하세요

합격선은 써도 되는지를 알려 주지만 어디가 고장 났는지는 알려 주지 않습니다. 교안 02 6-4처럼 기준선과 비교해 떨어진 지표로 고장 단계를 좁힙니다(상대 판정). 지표마다 기준선과의 차이를 한 표로 계산해 두면, 판정 순서표를 위에서부터 읽기만 하면 됩니다.

**판정 순서표** (교안 02 6-4와 같음): 차이는 "기준선 값 - 이 설정 값"이고, 위에서부터 보고 차이가 `0.2` 이상인 첫 지표의 진단 이름 하나만 붙입니다. 검색 지표 셋이 생성 지표 셋보다 앞에 있는 이유는, 필요한 문서가 빠지면 답변도 함께 나빠지기 때문입니다. 이 설정의 점수가 기준선보다 높은 지표는 차이가 음수라 해당하지 않습니다.

| 순서 | 지표(차이가 0.2 이상) | 진단 이름 | 뜻 |
|---|---|---|---|
| 1 | Context Recall | `검색 누락` | 필요한 문서(내용)를 가져오지 못함 |
| 2 | Context Relevancy | `엉뚱한 검색` | 질문과 관련 없는 문서를 가져옴 |
| 3 | Context Precision | `순위 문제` | 쓸모 있는 문서를 가져왔지만 아래 순위에 있음 |
| 4 | Faithfulness | `생성 환각` | 검색 문서에 없는 내용을 지어냄 |
| 5 | Answer Relevancy | `질문 이탈` | 문서대로 말했지만 질문이 묻는 내용이 아님 |
| 6 | Answer Correctness | `답 불일치` | 다른 지표는 괜찮은데 답이 정답 답변과 다름(문서가 낡았거나 답을 뽑지 못함) |
| 7 | 해당 없음 | `기준선과 비슷` | |

Answer Relevancy는 답을 피한 답변("안내문에서 확인할 수 없다")에 0점을 줍니다. 질문 이탈이 붙거나 Answer Relevancy 차이가 크면, 그 질문의 답변이 거절인지 먼저 봅니다.

**작성:** `make_diff_table(table)` 함수와 `diff_table`, 그 아래 마크다운 셀의 현재 운영 진단

**목적:** 8번에서 세 설정을 다시 볼 때도 쓰도록 차이 표를 함수로 만들고, 진단은 사람이 순서표로 읽어 적습니다.

**구현할 처리**
1. 종합 표의 기준선 행(`table.loc["기준선"]`)에서 종합 표 전체를 뺍니다. 행 하나(Series)에서 표(DataFrame)를 빼면 같은 열끼리 설정마다 빼집니다.
2. 소수 둘째 자리로 반올림해 반환합니다. 반올림한 평균끼리 빼도 컴퓨터의 실수 계산 오차 때문에 `1.00 - 0.80`이 `0.19999…`가 되므로, 반올림해야 차이가 정확히 0.2인 칸을 0.2로 읽을 수 있습니다.
3. 아래 마크다운 셀에 현재 운영의 진단 이름과 근거 지표(그 지표의 차이)를 씁니다.

**반환값:** 종합 표와 같은 모양의 차이 표(기준선 행은 모두 0)

In [ ]:
# [작성] 종합 표의 기준선 행에서 각 설정 행을 뺀 차이 표를 반환하는 함수를 완성하세요.
def make_diff_table(table):
    """종합 표의 설정마다 '기준선 값 - 이 설정 값'을 소수 둘째 자리로 반올림한 차이 표를 반환합니다."""
    # 1. 기준선 행에서 종합 표를 뺍니다 (같은 열끼리 설정마다 뺍니다)
    ...

    # 2. 소수 둘째 자리로 반올림해 반환
    ...


diff_table = make_diff_table(overall_table)
display(diff_table)

In [ ]:
# [자가채점] 가짜 종합 표로 방향·반올림 경계를 확인하고, 실제 차이 표는 종합 표에서 다시 계산해 대조합니다.
fake_table = pd.DataFrame(
    {"Answer Correctness": [0.7, 0.8], "Context Recall": [0.8, 1.0], "Context Precision": [1.0, 0.95],
     "Context Relevancy": [0.9, 1.0], "Faithfulness": [0.9, 0.9], "Answer Relevancy": [0.6, 0.8]},
    index=["가 설정", "기준선"],
)
fake_diff = make_diff_table(fake_table)
assert isinstance(fake_diff, pd.DataFrame), "차이 표(DataFrame)를 return 하세요. `...`를 채웠는지도 확인하세요."
assert list(fake_diff.columns) == list(fake_table.columns) and list(fake_diff.index) == list(fake_table.index), "차이 표는 종합 표와 같은 행·열 순서입니다."
assert fake_diff.loc["가 설정", "Context Precision"] == -0.05, "차이는 '기준선 값 - 이 설정 값'입니다. 기준선보다 높아진 지표는 음수입니다."
assert fake_diff.loc["가 설정", "Context Recall"] == 0.2, "1.00 - 0.80은 반올림해야 0.2입니다. 소수 둘째 자리로 반올림(round(2))하세요."
assert fake_diff.loc["기준선"].abs().sum() == 0, "기준선 행의 차이는 모두 0입니다."

expected = (overall_table.loc["기준선"] - overall_table).round(2)
for config in overall_table.index:
    for metric_name in METRIC_NAMES:
        assert abs(diff_table.loc[config, metric_name] - expected.loc[config, metric_name]) < 1e-9, f"{config} {metric_name}: diff_table은 make_diff_table(overall_table)의 결과입니다."
print("차이 표 확인 완료")

**현재 운영 진단** (판정 순서표를 위에서부터 읽어 작성)

- 진단 이름: (`검색 누락`·`엉뚱한 검색`·`순위 문제`·`생성 환각`·`질문 이탈`·`답 불일치`·`기준선과 비슷` 가운데 하나)
- 근거 지표와 차이: (차이가 0.2 이상인 첫 지표와 그 차이. 없으면 "없음")
- 그 위의 지표: (순서표에서 근거 지표보다 위에 있는 지표들의 차이가 0.2보다 작은지 한 줄로)

## 7. 검색 문서 원문으로 원인을 확인하세요

진단은 고장 위치를 가리킬 뿐이고, 같은 진단 이름도 원인은 여러 가지일 수 있습니다. 예를 들어 Context Recall이 떨어지는 원인은 필요한 문서를 못 가져온 것(검색기)일 수도, 문서는 가져왔지만 본문을 덜 넘긴 것일 수도 있습니다. 두 설정의 검색 문서를 나란히 놓고 어느 쪽인지 확인합니다.

**작성:** `cause_rows`(질문마다 한 행)

**목적:** 교안 02의 원문 확인처럼 진단을 검색 문서로 확인합니다. 문서 ID가 같은지, 정답 문서를 모두 가져왔는지, 넘긴 본문 길이를 함께 보면 고칠 곳이 검색기(`k`)인지 본문 자르기(`char_limit`)인지 가를 수 있습니다.

**구현할 처리**
1. 골격의 `base_runs`와 `current_runs`는 질문 순서가 같은 두 설정의 실행 목록입니다. `zip`으로 골든셋 질문과 같은 질문의 두 실행을 짝지어 읽습니다.
2. 두 실행의 `retrieved_ids`가 같은 ID를 같은 순서로 담았는지(`==`) 확인합니다.
3. 현재 운영 실행이 그 질문의 정답 문서(`relevant_doc_ids`)를 모두 가져왔는지 확인합니다. `set(정답 문서 목록).issubset(검색 문서 ID 목록)`은 정답 문서가 모두 검색 문서 ID 안에 있으면 `True`입니다. 앞 단원의 Recall@K처럼 문서 ID로 세는 검사입니다.
4. 두 실행이 넘긴 본문 글자 수를 셉니다. 검색 문서 본문 목록을 `"".join(...)`으로 이어 붙인 문자열의 길이입니다.
5. 행의 키는 `질문 ID`, `검색 문서 ID 같음`, `정답 문서 모두 포함`, `기준선 본문 글자 수`, `현재 운영 본문 글자 수`를 이 순서로 씁니다.

**반환값:** `cause_rows`(dict 6개 목록)

In [ ]:
# [작성] 질문마다 두 설정의 검색 문서 ID가 같은지와 넘긴 본문 글자 수를 한 행으로 만드세요.
# 1. 질문 순서가 같은 두 설정의 실행 목록 (generate_runs가 설정마다 questions 순서대로 쌓았습니다)
base_runs = runs[:len(questions)]
current_runs = runs[len(questions):]

cause_rows = []
for entry, base_run, current_run in zip(questions, base_runs, current_runs):
    # 2. 두 설정이 같은 검색 문서 ID를 같은 순서로 가져왔는지
    same_ids = ...

    # 3. 현재 운영이 정답 문서를 모두 가져왔는지 (문서 ID로 세는 검사)
    all_relevant = ...

    # 4~5. 넘긴 본문 글자 수를 세어 한 행을 더하세요.
    ...

display(pd.DataFrame(cause_rows))

In [ ]:
# [자가채점] 행 수·키·순서와 칸 값을 실행 목록에서 다시 계산해 대조합니다.
cause_keys = ["질문 ID", "검색 문서 ID 같음", "정답 문서 모두 포함", "기준선 본문 글자 수", "현재 운영 본문 글자 수"]
assert isinstance(cause_rows, list) and len(cause_rows) == len(questions), "cause_rows에는 질문마다 한 행, 모두 6행을 담으세요."
for row, entry, base_run, current_run in zip(cause_rows, questions, runs[:len(questions)], runs[len(questions):]):
    assert isinstance(row, dict) and list(row) == cause_keys, f"행의 키는 {cause_keys}를 이 순서로 씁니다."
    assert row["질문 ID"] == entry["question_id"], "행은 questions 순서대로 쌓으세요."
    assert row["검색 문서 ID 같음"] == (base_run["retrieved_ids"] == current_run["retrieved_ids"]), "검색 문서 ID 같음은 두 retrieved_ids 목록을 ==로 비교한 True/False입니다."
    assert row["정답 문서 모두 포함"] == set(entry["relevant_doc_ids"]).issubset(current_run["retrieved_ids"]), "정답 문서 모두 포함은 현재 운영 실행의 retrieved_ids에 그 질문의 relevant_doc_ids가 모두 있으면 True입니다."
    assert row["기준선 본문 글자 수"] == len("".join(base_run["retrieved_contexts"])), "글자 수는 검색 문서 본문 목록을 이어 붙인 문자열의 길이입니다."
    assert row["현재 운영 본문 글자 수"] == len("".join(current_run["retrieved_contexts"])), "글자 수는 검색 문서 본문 목록을 이어 붙인 문자열의 길이입니다."
print("원인 표 확인 완료:", len(cause_rows), "행")

골든셋에는 정답 답변의 주장마다 근거 문서(`doc_id`)와 근거 문장(`evidence`)이 적혀 있습니다. 아래 셀은 주장마다 근거 문장이 근거 문서 본문(제목 포함)의 몇 번째 글자에서 시작하는지(`근거 시작 위치`, 0부터 셈)와, 두 설정의 검색 문서 안에 그대로 들어 있는지 보여 줍니다. 근거 시작 위치를 `SETTINGS`의 `char_limit`와 비교해 보세요.

In [ ]:
# 주장마다 근거 문장이 두 설정의 검색 문서(넘긴 본문) 안에 있는지 표로 보기
evidence_rows = []
for entry, base_run, current_run in zip(questions, runs[:len(questions)], runs[len(questions):]):
    for claim in entry["claims"]:
        evidence_rows.append({
            "질문 ID": entry["question_id"],
            "근거 문서": claim["doc_id"],
            "근거 시작 위치": documents_by_id[claim["doc_id"]].find(claim["evidence"]),  # 근거 문서 본문에서 근거 문장이 시작하는 글자 위치
            "기준선에 있음": claim["evidence"] in "\n".join(base_run["retrieved_contexts"]),
            "현재 운영에 있음": claim["evidence"] in "\n".join(current_run["retrieved_contexts"]),
            "근거 문장": claim["evidence"][:50],
        })
display(pd.DataFrame(evidence_rows))

두 표와 종합 표를 함께 읽고 다음을 정리해 두세요. 결정 기록의 근거가 됩니다.

- 두 설정의 검색 문서 ID가 모든 질문에서 같다면, 검색기(`k`, 임베딩)는 같은 문서를 가져왔고 차이는 넘긴 본문에서 생긴 것입니다. 앞 단원의 Recall@K처럼 문서 ID로만 세는 지표는 이 차이를 보지 못합니다.
- 정답 문서를 모두 가져왔는데도 Context Recall이 떨어졌다면, 앞 단원의 Recall@K처럼 문서 ID로 세는 검사는 통과하고 본문 내용을 읽는 RAGAS 지표만 고장을 잡은 것입니다.
- 근거 표에서 현재 운영에 없는 근거 문장의 시작 위치와 `char_limit`를 비교합니다.
- 운영 팀이 바꾼 두 변경 가운데 어느 쪽이 진단이 가리킨 단계(검색 문서 또는 답변 생성)에 해당하는지 정합니다.

## 8. 진단이 가리킨 설정 하나만 되돌리고 다시 평가하세요

현재 운영은 `char_limit`와 프롬프트 두 설정이 함께 바뀌었습니다. 6~7번이 가리킨 설정 하나만 기준선 값으로 되돌리면, 남은 차이는 다른 변경 하나뿐이라 그 변경도 기준선과 비교해 따로 판단할 수 있습니다. 프롬프트를 바꾸는 것도 설정 하나를 바꾸는 것이므로, 되돌릴 대상이 프롬프트일 수도 있습니다. 비용을 줄이려던 운영 팀의 변경 가운데 고장과 관계없는 것은 남겨 둡니다.

**작성:** 작성 셀 두 개. 먼저 `SETTINGS["고친 운영"]`을 만들고, 설정을 자가채점으로 먼저 확인한 뒤 평가합니다. 그다음 `fixed_runs`, `fixed_rows`, `full_table`을 만듭니다.

**목적:** 진단 이름과 원인 확인이 함께 가리킨 설정 하나만 기준선 값으로 되돌린 설정을 만들고, 2~4번의 함수와 같은 방법으로 다시 평가합니다.

**구현할 처리**
1. `SETTINGS["고친 운영"]`은 `현재 운영`과 같은 세 키(`k`, `char_limit`, `chain`)를 이 순서로 쓴 dict입니다. 6~7번의 결과가 가리킨 설정 하나만 `기준선`의 값으로 바꾸고, 나머지 두 키는 `현재 운영`의 값을 그대로 씁니다.
2. `generate_runs`로 `고친 운영` 설정의 실행 6개를 만듭니다.
3. `evaluate_runs`로 그 실행을 평가해 점수 행 36개를 받습니다.
4. 3번 절에서 만든 `score_rows`와 이번 점수 행을 합쳐, 4번과 같은 방법으로 세 설정의 종합 표 `full_table`을 만듭니다. 행은 `["기준선", "현재 운영", "고친 운영"]`, 열은 `METRIC_NAMES` 순서로 고릅니다(`표.loc[행 목록, 열 목록]`).

**반환값:** 설정 하나가 더해진 `SETTINGS`, `fixed_runs`(실행 6개), `fixed_rows`(점수 행 36개), `full_table`(행 3개 × 열 6개)

In [ ]:
# [작성] 진단이 가리킨 설정 하나만 기준선 값으로 되돌린 "고친 운영" 설정을 만드세요.
# 1. 진단이 가리킨 키 하나만 기준선 값, 되돌리지 않은 나머지 두 키는 현재 운영 값 그대로
SETTINGS["고친 운영"] = ...

In [ ]:
# [자가채점] 고친 설정 값은 답이 보이지 않게 지문(sha256 앞 10자리)으로 대조합니다. API는 부르지 않습니다.
import hashlib


def fingerprint(value):
    """값의 repr을 sha256으로 바꾼 앞 10자리를 반환합니다."""
    return hashlib.sha256(repr(value).encode("utf-8")).hexdigest()[:10]


fixed_setting = SETTINGS["고친 운영"]
assert isinstance(fixed_setting, dict) and list(fixed_setting) == ["k", "char_limit", "chain"], "SETTINGS['고친 운영']은 k, char_limit, chain 세 키를 이 순서로 쓴 dict입니다."
setting_values = (fixed_setting["k"], fixed_setting["char_limit"], fixed_setting["chain"] is short_chain)
assert fingerprint(setting_values) == "09faf5ff9f", "6번 진단과 7번 원인 확인이 가리킨 설정 하나만 기준선 값으로 되돌리고, 되돌리지 않은 나머지 두 키는 현재 운영 값을 그대로 썼는지 확인하세요. 프롬프트(chain)도 설정 하나이고, chain에는 SETTINGS의 체인 객체를 그대로 쓰세요."
print("고친 운영 설정 확인 완료")

In [ ]:
# [작성] 고친 운영 설정으로 실행을 만들고 다시 평가해 세 설정의 종합 표를 만드세요.
# 2. 고친 운영으로 질문 6개의 실행 만들기
fixed_runs = ...

# 3. 지표 6개로 다시 평가 (evaluate_runs는 async def 함수라 await로 결과를 받습니다)
fixed_rows = ...

# 4. 세 설정의 종합 표 (행은 기준선·현재 운영·고친 운영, 열은 METRIC_NAMES 순서)
full_table = ...
display(full_table)

In [ ]:
# [자가채점] 실행·점수 행·표를 2~4번과 같은 규칙으로 확인합니다. LLM 점수 값 자체는 채점하지 않습니다.
check_runs(fixed_runs, ["고친 운영"])
check_score_rows(fixed_rows, fixed_runs)
check_table(full_table, score_rows + fixed_rows, ["기준선", "현재 운영", "고친 운영"])
assert list(full_table.index) == ["기준선", "현재 운영", "고친 운영"], "full_table의 행은 기준선, 현재 운영, 고친 운영 순서로 고르세요(표.loc[행 목록, 열 목록])."
print("고친 운영 확인 완료: 실행", len(fixed_runs), "개 | 점수 행", len(fixed_rows), "개")

In [ ]:
# 세 설정의 기준선 대비 차이 표 (6번에서 만든 함수)
display(make_diff_table(full_table))

**고친 운영 진단** (위 차이 표의 고친 운영 행을 판정 순서표로 읽어 작성)

- 진단 이름: (판정 순서표의 진단 이름 하나)
- 근거 지표와 차이: (없으면 "없음")
- 남긴 변경: (Faithfulness·Answer Relevancy·Answer Correctness 차이를 한 줄로)

In [ ]:
# 세 설정의 합격한 질문 수 (질문 6개 중, 기준선부터)
display(count_passes(score_rows + fixed_rows).loc[["기준선", "현재 운영", "고친 운영"]])

지표가 회복됐는지와 함께, 약사들이 알려 온 증상(안내문에서 확인할 수 없다는 답)도 줄었는지 답변 원문으로 확인합니다. 아래 두 셀은 API를 부르지 않습니다. 이 세기는 낱말만 찾으므로, 안내문에 근거가 없어 유보한 올바른 답도 함께 세어집니다. 숫자만 보지 말고 답변 원문을 읽으세요.

In [ ]:
# 설정별로 답변에 "확인할 수 없" 또는 "확인되지 않"이 들어간 실행 수
symptom_rows = []
for config in ["기준선", "현재 운영", "고친 운영"]:
    count = 0
    for run in runs + fixed_runs:
        if run["config"] == config:
            if "확인할 수 없" in run["response"]:
                count += 1
            elif "확인되지 않" in run["response"]:
                count += 1
    symptom_rows.append({"설정": config, "확인할 수 없다는 답변 수": count, "실행 수": len(questions)})
display(pd.DataFrame(symptom_rows))

In [ ]:
# 질문마다 기준선과 고친 운영의 답변 원문을 정답 답변과 나란히 출력 (정답 답변과 나란히 읽어 빠진 내용이 있는지 봅니다)
for base_run, fixed_run in zip(runs[:len(questions)], fixed_runs):
    print(base_run["question_id"], "| 기준선:", base_run["response"], "| 고친 운영:", fixed_run["response"], "| 정답 답변:", base_run["reference"])

`고친 운영`의 진단이 `기준선과 비슷`이고 근거 지표였던 열이 기준선 수준으로 돌아왔다면, 진단이 가리킨 설정이 원인이었다는 뜻입니다. 진단이 다르게 나왔다면 질문별 점수표에서 평균을 끌어내린 질문을 찾아 원문을 확인하고, 평가 셀을 한 번 더 실행해 같은 진단이 나오는지 봅니다.

`고친 운영`과 기준선의 차이는 남겨 둔 변경 하나뿐입니다. 남긴 변경을 그대로 둘지는 아래 세 가지로 정합니다.

1. **진단:** `고친 운영`이 `기준선과 비슷`이어야 합니다. 남긴 변경이 답변 내용을 빠뜨리게 했다면 `답 불일치`(Answer Correctness), 질문에서 엇나가게 했다면 `질문 이탈`(Answer Relevancy)이 붙습니다.
2. **합격 질문 수:** 합격선 지표마다 `고친 운영`의 합격 질문 수가 기준선보다 2개 이상 줄지 않아야 합니다. 질문이 6개라 1개 차이는 평가용 LLM 판정이 흔들리는 범위로 봅니다. 2개 이상 줄었다면 교안 02 6-6처럼 평가 셀을 한 번 더 실행하고, 다시 2개 이상 줄 때만 남긴 변경 때문으로 봅니다.
3. **원문 확인:** 의약품처럼 틀리면 피해가 큰 분야에서는 지표만으로 끝내지 않습니다. 위 답변 원문에서 질문이 물은 용량·횟수·대상 연령·함께 먹으면 안 되는 약 같은 안전 정보를 `고친 운영` 답변이 빠뜨렸는지 정답 답변과 대조하고, 기준선 답변도 같은 정보를 빠뜨렸는지 함께 봅니다.

1·2를 통과해도 3에서 안전 정보가 빠졌다면 남긴 변경도 되돌릴지 근거와 함께 판단해, 아래 칸과 결정 기록에 씁니다.

**남긴 변경 원문 확인** (위 답변 원문과 정답 답변을 대조해 작성)

- 고친 운영 답변이 빠뜨린 안전 정보: (질문 ID와 빠진 용량·횟수·대상 연령·함께 먹으면 안 되는 약. 없으면 "없음")
- 기준선 답변도 같은 정보를 빠뜨렸는지: (질문 ID별로 한 줄)
- 판단: (남긴 변경을 유지할지 되돌릴지와 그 근거 한두 문장)

## 9. 결정 기록을 쓰세요

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 채웁니다.

- **제목:** 고른 조치가 드러나게 씁니다(예: "○○를 △△로 변경").
- **맥락:** 정해야 할 문제·조건·평가 방법을 2~3문장으로 쓰고 결정은 쓰지 않습니다.
- **결정:** "…한다."로 시작하고, 아래 `근거:`에 종합 표·합격 표·차이 표·원인 표·근거 표·답변 원문의 숫자와 사실을 한 줄씩 씁니다.
- **결과:** `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(바뀌는 것 -> 다시 할 일)을 씁니다.
- **문체:** 서비스 문서처럼 과제·절 번호 없이 사실을 직접 씁니다.
- **숫자:** 자신이 실행한 표와 출력에서 옮깁니다. 점수는 실행마다 조금씩 달라지므로, 정답 예시와 숫자나 원문 확인 결과가 다르면 자기 결과로 씁니다.

**목적:** 합격선으로 센 합격 질문 수(써도 되는가), 기준선 비교 진단(어디가 고장 났는가), 답변 원문 확인(안전 정보가 빠졌는가)을 함께 근거로, 두 변경 가운데 무엇을 되돌리고 무엇을 남기는지 설명하고 결론이 바뀌는 조건을 남깁니다.

**설명할 내용**
1. 상태: 제안·채택·거절·대체됨 가운데 하나(이 결정을 받아들였으면 `채택`)
2. 맥락: 무엇이 문제였는지(약사 문의), 운영 팀이 바꾼 두 변경, 평가 방법(골든셋 6문항, 지표 6개, 합격선, 기준선과의 차이 0.2, 답변 원문 확인)
3. 결정: 되돌리는 변경과 남기는 변경을 한 문장으로 쓰고, `근거:`에 다음을 한 줄씩 씁니다.
    - 현재 운영의 합격 질문 수와 진단, 근거 지표의 기준선·현재 운영 평균
    - 검색 문서 ID와 정답 문서 포함 여부, 넘긴 본문 글자 수, 현재 운영에서 빠진 근거 문장의 시작 위치
    - 고친 운영의 진단과 근거 지표였던 열의 평균(회복했는지), 합격 질문 수, 확인할 수 없다는 답변 수의 변화
    - 남긴 변경을 판단한 근거: 8번의 세 가지(진단, 합격 질문 수가 기준선보다 2개 이상 줄었는지, 답변 원문에서 빠진 안전 정보)
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`을 둘 이상(답변 품질 지표, 안전 정보 누락, 문서 길이, 비용 등)

#### 결정 기록: (고른 조치가 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2~3문장: 문제(약사 문의), 운영 팀이 바꾼 두 변경, 평가 방법. 결정은 쓰지 않음)
- **결정**: (되돌리는 변경과 남기는 변경을 한 문장으로: …한다)
    - 근거: (한 줄씩: 현재 운영의 합격 질문 수와 진단 숫자, 검색 문서 ID·정답 문서 포함·본문 글자 수·빠진 근거의 시작 위치, 고친 운영의 진단과 회복한 숫자·합격 질문 수·확인할 수 없다는 답변 수, 남긴 변경의 진단·합격 질문 수·답변 원문에서 빠진 안전 정보)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 가정이나 수치 -> 다시 할 일)